Jose Miguel Armas

Juan David Salazar

Juan Jose Vidarte

Santiago Zapata

In [8]:
!pip install pyformlang

# Exercise 1

In [9]:
from pyformlang.fst import FST

ar_endings = {
    'yo': 'o', 'tu': 'as', 'el': 'a', 'ella': 'a',
    'nosotros': 'amos', 'nosotras': 'amos',
    'vosotros': 'ais',  'vosotras': 'ais',
    'ellos': 'an',      'ellas': 'an'
}
er_endings = {
    'yo': 'o', 'tu': 'es', 'el': 'e', 'ella': 'e',
    'nosotros': 'emos', 'nosotras': 'emos',
    'vosotros': 'eis',  'vosotras': 'eis',
    'ellos': 'en',      'ellas': 'en'
}

verbs = {
    'comer':  ('com',  er_endings),
    'cantar': ('cant', ar_endings),
    'bailar': ('bail', ar_endings),
}

pronouns = list(ar_endings.keys())

fst1 = FST()
fst1.add_start_state('q0')

for pronoun in pronouns:
    q_p = f'qp_{pronoun}'
    q_s = f'qs_{pronoun}'
    fst1.add_transitions([('q0', pronoun, q_p, [pronoun])])
    fst1.add_transitions([(q_p, ' ', q_s, [' '])])
    for verb, (stem, endings) in verbs.items():
        q_f = f'qf_{pronoun}_{verb}'
        conjugated = stem + endings[pronoun]
        fst1.add_transitions([(q_s, verb, q_f, [conjugated])])
        fst1.add_final_state(q_f)

test_cases = [
    'yo comer', 'tu comer', 'el comer',
    'ella comer', 'nosotros comer', 'ellos comer',
    'yo cantar', 'tu cantar', 'ella bailar',
    'vosotros bailar', 'ellas cantar',
]

for inp in test_cases:
    tokens = inp.split(' ')
    tokens_with_space = [tokens[0], ' ', tokens[1]]

    results = list(fst1.translate(tokens_with_space))
    output = " ".join(results[0]) if results else "(no translation)"
    print(f"'{inp}' -> '{output}'")

'yo comer' -> 'yo   como'
'tu comer' -> 'tu   comes'
'el comer' -> 'el   come'
'ella comer' -> 'ella   come'
'nosotros comer' -> 'nosotros   comemos'
'ellos comer' -> 'ellos   comen'
'yo cantar' -> 'yo   canto'
'tu cantar' -> 'tu   cantas'
'ella bailar' -> 'ella   baila'
'vosotros bailar' -> 'vosotros   bailais'
'ellas cantar' -> 'ellas   cantan'


#Exercise 2

In [10]:
from pyformlang.fst import FST

past_forms = {
    'to eat':   'ate',
    'to make':  'made',
    'to mount': 'mounted',
    'to end':   'ended',
}

fst2 = FST()
fst2.add_start_state('q0')

for i, (infinitive, past) in enumerate(past_forms.items()):
    q_f = f'qf{i}'
    fst2.add_transitions([('q0', infinitive, q_f, [past])])
    fst2.add_final_state(q_f)

for inf in past_forms:
    results = list(fst2.translate([inf]))
    output = results[0][0] if results else "(no translation)"
    print(f"'{inf}' -> '{output}'")


'to eat' -> 'ate'
'to make' -> 'made'
'to mount' -> 'mounted'
'to end' -> 'ended'


# Exercise 22

The language accepts strings with: one `a`, then one or more `b`'s, then zero or more `a`'s.

**Examples:** `ab`, `abb`, `aba`, `abba`, `abbaa`

---

### NFA
```
q0 --a--> q1 --b--> q2
                    q2 --b--> q2
                    q2 --a--> q2
                    q2 = final state
```

---

### Grammar derived from the NFA

Each state becomes a non-terminal, each transition becomes a rule, and each final state gets an $\varepsilon$ rule.

$$S \to aA$$
$$A \to bB$$
$$B \to bB \mid aB \mid \varepsilon$$

- $S \to aA$ : reads the mandatory `a`
- $A \to bB$ : reads at least one `b`
- $B \to bB \mid aB$ : keeps reading `b`'s or `a`'s
- $B \to \varepsilon$ : stops whenever we want

In [11]:
from pyformlang.cfg import CFG

cfg22 = CFG.from_text("""
S -> a A
A -> b B
B -> b B
B -> a B
B -> epsilon
""")

tests = [
    ('ab',     True),   # a + b(1) + a*(0)
    ('abb',    True),   # a + b(2) + a*(0)
    ('aba',    True),   # a + b(1) + a*(1)
    ('abba',   True),   # a + b(2) + a*(1)
    ('abbaa',  True),   # a + b(2) + a*(2)
    ('abbbba', True),   # a + b(4) + a*(1)
    ('a',      False),  # missing b
    ('b',      False),  # no leading a
    ('ba',     False),  # wrong order
    ('aab',    False),  # two a's before b
    ('',       False),  # empty string
]

for word, expected in tests:
    result = cfg22.contains(word)
    status = "PASS" if result == expected else "FAIL"
    print(f"[{status}] '{word}' -> {result}  (expected {expected})")

[PASS] 'ab' -> True  (expected True)
[PASS] 'abb' -> True  (expected True)
[PASS] 'aba' -> True  (expected True)
[PASS] 'abba' -> True  (expected True)
[PASS] 'abbaa' -> True  (expected True)
[PASS] 'abbbba' -> True  (expected True)
[PASS] 'a' -> False  (expected False)
[PASS] 'b' -> False  (expected False)
[PASS] 'ba' -> False  (expected False)
[PASS] 'aab' -> False  (expected False)
[PASS] '' -> False  (expected False)


# Exercise 24




The word is made of 4 parts, each controlled by a counter $i$, $j$ or $k$:

$$\underbrace{b^i a b^{2i}}_{\text{part 1, counter } i} \quad \underbrace{a^{2j+1}}_{\text{part 2, counter } j} \quad \underbrace{c^k b^{2k}}_{\text{part 3, counter } k} \quad \underbrace{a^j}_{\text{part 4, counter } j}$$

We build one sub-grammar per counter.

---

**P** handles $b^i a b^{2i}$. Each recursion wraps the word with 1 `b` on the left and 2 `b`'s on the right, so both sides grow at the right rate:
$$P \to bPbb \mid babb$$

**N** handles $c^k b^{2k}$. Same idea, but with `c` on the left and 2 `b`'s on the right:
$$N \to cNbb \mid cbb$$

**Q** handles parts 2 and 4 together ($a^{2j+1}$ before $N$ and $a^j$ after). Each recursion adds 2 `a`'s on the left and 1 on the right, so the left always grows twice as fast. The base case manually sets $j=1$:
$$Q \to aaQa \mid aaaNa$$

We can verify this works:
- $j=1$: $aaaNa = a^3 N a^1$ ✓
- $j=2$: $aa(aaaNa)a = a^5 N a^2$ ✓
- $j=3$: $aa(a^5Na^2)a = a^7 N a^3$ ✓

---

**Full Grammar**

$$S \to PQ$$
$$P \to bPbb \mid babb$$
$$Q \to aaQa \mid aaaNa$$
$$N \to cNbb \mid cbb$$

---

Example: $i=1, \ j=1, \ k=1$
```
P => babb
N => cbb
Q => aaa(cbb)a = aaacbba

S => babb + aaacbba = 'babbaaacbba'
```

In [12]:
from pyformlang.cfg import CFG

cfg24 = CFG.from_text("""
S -> P Q
P -> b P b b
P -> b a b b
Q -> a a Q a
Q -> a a a N a
N -> c N b b
N -> c b b
""")

def make_word(i, j, k):
    """Build a word in L with parameters i, j, k."""
    return 'b'*i + 'a' + 'b'*(2*i) + 'a'*(2*j+1) + 'c'*k + 'b'*(2*k) + 'a'*j

print("Words IN L:")
in_L = [(1,1,1),(1,1,2),(2,1,1),(1,2,1),(2,2,2),(3,1,1),(1,3,1)]
for (i,j,k) in in_L:
    word = make_word(i,j,k)
    result = cfg24.contains(word)
    status = "PASS" if result else "FAIL"
    print(f"  [{status}] i={i} j={j} k={k}  '{word}' -> {result}")

print("\nWords NOT in L:")
not_L = [
    ('babbaaacb',   "c^1 b^1 instead of c^1 b^2"),
    ('aaa',         "no b or c structure"),
    ('bacb',        "wrong symbol order"),
    ('babbaaaabb',  "missing c segment"),
    ('bbbaaacbb',   "b^3 a b^2 violates b^(2i) for any i"),
]
for word, reason in not_L:
    result = cfg24.contains(word)
    status = "PASS" if not result else "FAIL"
    print(f"  [{status}] '{word}' -> {result}  ({reason})")

Words IN L:
  [PASS] i=1 j=1 k=1  'babbaaacbba' -> True
  [PASS] i=1 j=1 k=2  'babbaaaccbbbba' -> True
  [PASS] i=2 j=1 k=1  'bbabbbbaaacbba' -> True
  [PASS] i=1 j=2 k=1  'babbaaaaacbbaa' -> True
  [PASS] i=2 j=2 k=2  'bbabbbbaaaaaccbbbbaa' -> True
  [PASS] i=3 j=1 k=1  'bbbabbbbbbaaacbba' -> True
  [PASS] i=1 j=3 k=1  'babbaaaaaaacbbaaa' -> True

Words NOT in L:
  [PASS] 'babbaaacb' -> False  (c^1 b^1 instead of c^1 b^2)
  [PASS] 'aaa' -> False  (no b or c structure)
  [PASS] 'bacb' -> False  (wrong symbol order)
  [PASS] 'babbaaaabb' -> False  (missing c segment)
  [PASS] 'bbbaaacbb' -> False  (b^3 a b^2 violates b^(2i) for any i)
